# Example 11.4 / 2: Starting Points and Local Optima

Companion notebook for **11.4 Pitfalls of Nonlinear Programming** in the AMPL lab Documentation, section "Starting Points and Local Optima". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Starting Points and Local Optima

In a nonconvex NLP, different starting points can lead a local solver to different local optima. Multi-start experiments can find a better solution, but sampling several starts is not a proof of global optimality.

![A nonconvex objective curve with two starting points converging to different local minima and the lower one marked global](/assets/docs/local-global-optima.svg)

*A local method follows nearby descent geometry. Different starts can enter different basins of attraction, and only the lowest minimum over the full domain is globally optimal.*

The objective $(x^2-1)^2+0.1x$ has two wells, near $x=-1$ and $x=1$. Ipopt started in each well
stays in it, and both runs report `solved`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= -2, <= 2;                      # finite bounds
minimize Wells: (x^2 - 1)^2 + 0.1 * x;  # nonconvex: two local minima

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
for start in (-1.5, 1.5):
    ampl.var["x"].set_value(start)
    ampl.solve(solver="ipopt", verbose=False)
    print(start, ampl.solve_result, round(ampl.var["x"].value(), 6),
          round(ampl.obj["Wells"].value(), 6))

-1.5 solved -1.012273 -0.100617
1.5 solved 0.987257 0.099367


### Pure AMPL: model.run

Pure AMPL, with a `for` loop and `printf`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver ipopt;
for {start in {-1.5, 1.5}} {
    let x := start;
    solve > ipopt.log;
    printf "%g %s %.6f %.6f\n", start, solve_result, x, Wells;
}

 
Ipopt 3.12.13: Optimal Solution Found

suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
-1.5 solved -1.012273 -0.100617
 
Ipopt 3.12.13: Optimal Solution Found
1.5 solved 0.987257 0.099367
